# Summary Combo — 3-Workbook Generator

Builds three workbooks from the October week-3 source files:

1. **`OCT-WK3-DATEWISE.xlsx`** — 5 sheets (`crates`, `mehul_returns`, `hansa_returns`, `sales_revenue`, `brands_performance`), all dates, no precision, no pivoting.
2. **`OCT-WK3-PRECISE.xlsx`** — 5 sheets, one row per `A/C`, dates collapsed, products pivoted (`INV`/`CN` or `FRESH`/`EXP`).
3. **`OCT-WK3-COMBINED.xlsx`** — 1 sheet, all domains pivoted into one wide table with credits kept as separate columns.

Every sheet joins the master `customer-info.xlsx` on the A/C code so `CUSTOMER_NAME`, `CATEGORY`, `ROUTE_NAME`, `SALES_REP`, `MERCHANDISERS` are always present.

In [14]:
import os
import re
import warnings
import pandas as pd
import numpy as np

warnings.filterwarnings("ignore")

## Configuration

In [15]:
OUTPUT_DIR = "report/october"
os.makedirs(OUTPUT_DIR, exist_ok=True)

CUSTOMER_INFO_FILE = "data/customer-info/customer-info.xlsx"
BRANDS_FILE        = "data/brands/oct-week3-brand.xlsx"
CRATES_FILE        = "data/crates/oct-week3-crates.xlsx"
MEHUL_FILE         = "data/mehul-returns/OCTOBER_RETURNS.xlsx"
HANSA_FILE         = "data/hansa-returns/oct-week3-hansa-returns.xlsx"
SALES_FILE         = r"data\sales-revenue\october-week3-sales-revenue.xlsx"

# Canonical brand list (drives output column ordering)
BRANDS = [
    "P", "S", "B", "200G", "O.P.", "800G", "600G",
    "MANDAZI", "DOUGHNUTS", "MANDO FAMILY PACK", "4 SQUARE",
    "XTRA LOONG", "B-BITEZ", "KATIKATI", "50-50 SLICED",
]

# Bread weights -> 400g equivalent (used only for brand ratio in combined workbook)
BREAD_FACTORS = {"200G": 0.5, "800G": 2.0, "600G": 1.5}

## Normalisers
Column names differ slightly across the six source files (`200g` vs `200G`, `O.P.` vs `O.P`, `4 Square` vs `4SQ`, `B-BITEZ` vs `BBITEZ`). Everything is normalised to a canonical form so joins and pivots stay consistent.

In [16]:
def norm_col(c: str) -> str:
    """Normalise a column name so it matches across all source files."""
    s = str(c).strip().upper()
    s = s.replace(".", " ").replace("&", " ").replace("-", " ").replace("/", " ")
    s = re.sub(r"\s+", " ", s)
    return s.strip()


def norm_brand(c: str) -> str:
    """Map a normalised column name to its canonical brand key (or '')."""
    n = norm_col(c)
    aliases = {
        "P": "P", "S": "S", "B": "B",
        "200G": "200G",
        "O P": "O.P.", "OP": "O.P.",
        "800G": "800G", "600G": "600G",
        "MANDAZI": "MANDAZI", "MANDO": "MANDAZI",
        "DOUGHNUTS": "DOUGHNUTS",
        "MANDO FAMILY PACK": "MANDO FAMILY PACK", "FAMILY PACK": "MANDO FAMILY PACK",
        "4 SQUARE": "4 SQUARE", "4SQ": "4 SQUARE",
        "XTRA LOONG": "XTRA LOONG", "XLOONG": "XTRA LOONG",
        "B BITEZ": "B-BITEZ", "BBITEZ": "B-BITEZ", "BB": "B-BITEZ",
        "KATIKATI": "KATIKATI",
        "50 50 SLICED": "50-50 SLICED", "5050 SLICED": "50-50 SLICED",
    }
    return aliases.get(n, "")


def norm_ac(series: pd.Series) -> pd.Series:
    """Normalise customer code so 55, 55.0 and '55' all match."""
    return pd.to_numeric(series, errors="coerce").astype("Int64").astype("string")


def safe_brand_name(brand: str) -> str:
    """Turn a canonical brand into a valid, consistent Excel column fragment."""
    return (brand.replace(" ", "_").replace(".", "").replace("-", "_"))

## Customer info (master lookup)

In [17]:
def load_customer_info() -> pd.DataFrame:
    ci = pd.read_excel(CUSTOMER_INFO_FILE)
    ci.columns = [c.strip().upper() for c in ci.columns]
    ci = ci.rename(columns={
        "CODE": "A/C",
        "CUSTOMER NAME": "CUSTOMER_NAME",
        "ROUTE NAME": "ROUTE_NAME",
    })
    ci["A/C"] = norm_ac(ci["A/C"])
    customer_code = ci["CUSTOMER_NAME"].astype("string").str.extract(r"\((\d+)\)\s*$")[0]
    duplicate_codes = ci.loc[ci["A/C"].duplicated(keep=False), "A/C"].unique()
    for ac in duplicate_codes:
        duplicates = ci[ci["A/C"].eq(ac)]
        code_match = customer_code.loc[duplicates.index].eq(ac).fillna(False)
        if code_match.sum() != 1:
            raise ValueError(f"Conflicting customer master rows for A/C {ac}")
        ci = ci.drop(index=duplicates.index[~code_match])
    ci = ci.drop_duplicates("A/C").reset_index(drop=True)
    return ci


CUST_COLS = ["A/C", "CUSTOMER_NAME", "CATEGORY", "ROUTE_NAME", "SALES_REP", "MERCHANDISERS"]


def attach_customer(df: pd.DataFrame, ac_col: str = "A/C") -> pd.DataFrame:
    """Left-join customer-info onto any dataframe using its A/C column."""
    ci = load_customer_info()
    df = df.copy()
    df[ac_col] = norm_ac(df[ac_col])
    drop_cols = [c for c in CUST_COLS[1:] if c in df.columns]
    df = df.drop(columns=drop_cols, errors="ignore")
    df = df.merge(ci, on=ac_col, how="left")
    return df

## Loaders — one per source file

### Brands — remove one matching credit per invoice (as in `BRANDS.ipynb`)

In [18]:
BRAND_NUMERIC_RAW = ["P","S","B","200g","O.P.","800g","1500g","ST","600g","KATIKATI",
                     "MANDAZI","DOUGHNUTS","50-50 SLICED","MANDO FAMILY PACK",
                     "4 Square","XTRA LOONG","B-BITEZ","Qty"]


def remove_one_invoice_per_credit(invoices, credit_notes, numeric_columns):
    invoices = invoices.copy()
    credit_notes = credit_notes.copy()
    invoices["is_credit"] = False
    used = set()
    for col in numeric_columns:
        if col in invoices.columns:
            invoices[col] = pd.to_numeric(invoices[col], errors="coerce")
        if col in credit_notes.columns:
            credit_notes[col] = pd.to_numeric(credit_notes[col], errors="coerce")
    for inv_idx, inv_row in invoices.iterrows():
        ac = inv_row["A/C"]
        cn_sub = credit_notes[(credit_notes["A/C"] == ac) & (~credit_notes.index.isin(used))]
        for cn_idx, cn_row in cn_sub.iterrows():
            match = True
            for col in numeric_columns:
                if col not in inv_row.index or col not in cn_row.index:
                    match = False; break
                a, b = inv_row[col], cn_row[col]
                if pd.isna(a) or pd.isna(b) or a != -b:
                    match = False; break
            if match:
                invoices.at[inv_idx, "is_credit"] = True
                used.add(cn_idx)
                break
    return invoices


def load_brands():
    inv = pd.read_excel(BRANDS_FILE, sheet_name="Brands-inv")
    cn  = pd.read_excel(BRANDS_FILE, sheet_name="Brands-cn")
    cols = [c for c in BRAND_NUMERIC_RAW if c in inv.columns and c in cn.columns]
    inv["A/C"] = norm_ac(inv["A/C"]); cn["A/C"] = norm_ac(cn["A/C"])
    inv["Date"] = pd.to_datetime(inv["Date"])
    cn["Date"]  = pd.to_datetime(cn["Date"])
    matched = remove_one_invoice_per_credit(inv, cn, cols)
    inv_clean = matched[~matched["is_credit"]].drop(columns=["is_credit"])
    return inv_clean, cn, cols

### Crates

In [19]:
def load_crates():
    inv = pd.read_excel(CRATES_FILE, sheet_name="Crates-inv")
    cn  = pd.read_excel(CRATES_FILE, sheet_name="Crates-cn")
    for df in (inv, cn):
        df.columns = df.columns.str.strip()
        df["A/C"] = norm_ac(df["A/C"])
        df["QTY"] = pd.to_numeric(df["QTY"], errors="coerce").fillna(0).astype(int)
        df["DATES"] = pd.to_datetime(df["DATES"])
    return inv, cn

### Mehul returns — every daily sheet (`1ST`, `2ND`, `3RD`, …)

In [20]:
MEHUL_DAILY_RE = re.compile(r"^(\d{1,2})(ST|ND|RD|TH)$", re.IGNORECASE)


def _parse_mehul_sheet(xl, sheet_name):
    raw = pd.read_excel(xl, sheet_name=sheet_name, header=None)
    if raw.shape[0] < 4 or raw.shape[1] < 6:
        return None
    base_cols = ["ROUTE_NO", "ROUTE", "CATEGORY", "CODE", "NAME"]
    brands_row = raw.iloc[1, 5:].tolist()
    types_row  = raw.iloc[2, 5:].tolist()
    source_cols = []
    current_brand = None
    for i, (b, t) in enumerate(zip(brands_row, types_row)):
        if pd.notna(b) and str(b).strip():
            current_brand = norm_col(b)
        tname = norm_col(t) if pd.notna(t) else ""
        tname = {"EX": "EXP", "1": "EXP", "1.0": "EXP"}.get(tname, tname)
        if current_brand == "TOTAL" and pd.notna(b):
            source_cols.append("TOTAL_ROUTE")
        elif current_brand and tname in ("FRESH", "EXP"):
            source_cols.append(f"{current_brand}_{tname}")
        else:
            source_cols.append(f"_IGNORED_{i}")
    data = raw.iloc[3:].copy()
    data.columns = base_cols + source_cols
    data = data.dropna(how="all")
    data = data[data["NAME"].astype("string").str.strip().str.upper() != "TOTAL"]
    for c in data.columns:
        if c == "TOTAL_ROUTE" or c.endswith("_FRESH") or c.endswith("_EXP"):
            data[c] = pd.to_numeric(data[c], errors="coerce").fillna(0)
    return data


def load_mehul():
    xl = pd.ExcelFile(MEHUL_FILE)
    daily = [s for s in xl.sheet_names if MEHUL_DAILY_RE.match(s.strip())]
    frames = []
    for s in daily:
        d = _parse_mehul_sheet(xl, s)
        if d is not None and len(d):
            d["_source_sheet"] = s
            frames.append(d)
    if not frames:
        return pd.DataFrame()
    df = pd.concat(frames, ignore_index=True)
    df["CODE"] = norm_ac(df["CODE"])
    df["ROUTE"] = df["ROUTE"].astype("string").fillna("").str.strip()
    df["NAME"]  = df["NAME"].astype("string").fillna("").str.strip()
    df["CATEGORY"] = df["CATEGORY"].astype("string").fillna("").str.strip()
    return df

### Hansa returns

In [21]:
def load_hansa():
    df = pd.read_excel(HANSA_FILE)
    df.columns = [c.strip() for c in df.columns]
    df["A/C"] = norm_ac(df["A/C"])
    df["Date"] = pd.to_datetime(df["Date"])
    df["Name"] = df["Name"].astype("string").str.replace("\xa0", " ", regex=False).str.strip()
    return df

### Sales revenue (October week 3 invoices + credits)

Revenue uses the weekly workbook's `Inv. Total` (amount plus VAT and extra cents), not its base `Amount` alone.

In [22]:
def load_sales_revenue():
    inv = pd.read_excel(SALES_FILE, sheet_name="Invoice")
    cn  = pd.read_excel(SALES_FILE, sheet_name="Credit")
    for df in (inv, cn):
        df.columns = (df.columns.str.strip().str.lower()
                      .str.replace(" ", "_").str.replace("(", "", regex=False)
                      .str.replace(")", "", regex=False).str.replace(".", "", regex=False))
        required = {"date", "a/c_no", "qty", "amount", "inv_total"}
        missing = required.difference(df.columns)
        if missing:
            raise ValueError(f"Missing sales columns in {SALES_FILE}: {sorted(missing)}")
        df["date"] = pd.to_datetime(df["date"], errors="raise")
        df["a/c_no"] = norm_ac(df["a/c_no"])
        df["qty"] = pd.to_numeric(df["qty"], errors="raise")
        df["amount_before_extra"] = pd.to_numeric(df["amount"], errors="raise")
        df["amount"] = pd.to_numeric(df["inv_total"], errors="raise")
        df.drop(columns=["inv_total"], inplace=True)
    return inv, cn

## Workbook 1 — `OCT-WK3-DATEWISE.xlsx`
5 sheets, all dates, no precision, no pivoting. Just raw rows joined to customer info.

In [23]:
def build_datewise(out_path):
    sheets = {}

    # ---------- crates ----------
    c_inv, c_cn = load_crates()
    c_inv = c_inv.assign(STATUS="Inv")
    c_cn  = c_cn.assign(STATUS="CN")
    crates_all = pd.concat([c_inv, c_cn], ignore_index=True)
    crates_all = crates_all.rename(columns={"DATES": "Date"})
    crates_all = attach_customer(crates_all)
    sheets["crates"] = crates_all[["Date", "STATUS"] + CUST_COLS + ["Invoice No.", "QTY"]]

    # ---------- mehul_returns ----------
    meh = load_mehul()
    if not meh.empty:
        meh = meh.rename(columns={"CODE": "A/C", "NAME": "NAME_RAW", "ROUTE": "ROUTE_NAME_RAW"})
        meh = attach_customer(meh)
        front = ["_source_sheet", "A/C"] + CUST_COLS[1:] + ["ROUTE_NO", "TOTAL_ROUTE"]
        other = [c for c in meh.columns if c not in front]
        meh = meh[front + other]
    sheets["mehul_returns"] = meh

    # ---------- hansa_returns ----------
    han = load_hansa()
    han = attach_customer(han)
    if "Name" in han.columns:
        han = han.rename(columns={"Name": "NAME_RAW"})
    front = ["Date", "A/C"] + CUST_COLS[1:] + ["TYPE", "Invoice", "Qty"]
    other = [c for c in han.columns if c not in front]
    han = han[front + other]
    sheets["hansa_returns"] = han

    # ---------- sales_revenue ----------
    s_inv, s_cn = load_sales_revenue()
    s_inv = s_inv.assign(STATUS="Inv")
    s_cn  = s_cn.assign(STATUS="CN")
    sales_all = pd.concat([s_inv, s_cn], ignore_index=True)
    sales_all = sales_all.rename(columns={"date": "Date", "a/c_no": "A/C"})
    sales_all = attach_customer(sales_all)
    front = ["Date", "STATUS", "A/C"] + CUST_COLS[1:] + ["invoice_no", "qty", "amount"]
    other = [c for c in sales_all.columns if c not in front]
    sales_all = sales_all[front + other]
    sheets["sales_revenue"] = sales_all

    # ---------- brands_performance ----------
    b_inv_clean, b_cn, _ = load_brands()
    b_inv_clean = b_inv_clean.assign(STATUS="Inv")
    b_cn        = b_cn.assign(STATUS="CN")
    brands_all = pd.concat([b_inv_clean, b_cn], ignore_index=True, sort=False)
    brands_all = attach_customer(brands_all)
    front = ["Date", "STATUS", "A/C"] + CUST_COLS[1:] + ["Invoice No.", "Qty"]
    other = [c for c in brands_all.columns if c not in front]
    brands_all = brands_all[front + other]
    sheets["brands_performance"] = brands_all

    with pd.ExcelWriter(out_path, engine="openpyxl") as writer:
        for name, df in sheets.items():
            df.to_excel(writer, sheet_name=name, index=False)
    print(f"[DATEWISE] wrote {out_path}")
    for n, df in sheets.items():
        print(f"  {n}: {len(df)} rows x {len(df.columns)} cols")
    return sheets

## Workbook 2 — `OCT-WK3-PRECISE.xlsx`
5 sheets, one row per `A/C`, dates collapsed, products pivoted.

In [24]:
def build_precise(out_path):
    sheets = {}

    # ---------- crates ----------
    c_inv, c_cn = load_crates()
    c_inv = c_inv.assign(STATUS="Inv")
    c_cn  = c_cn.assign(STATUS="CN")
    crates_all = pd.concat([c_inv, c_cn], ignore_index=True)
    crates_all = crates_all.rename(columns={"DATES": "Date"})
    crates_all = attach_customer(crates_all)
    crates_p = crates_all.groupby(CUST_COLS + ["STATUS"], as_index=False, dropna=False)["QTY"].sum()
    crates_p = crates_p.pivot(
        index=CUST_COLS, columns="STATUS", values="QTY"
    ).reset_index()
    crates_p = crates_p.rename(columns={"Inv": "CRATES_INV_QTY", "CN": "CRATES_CN_QTY"})
    for c in ("CRATES_INV_QTY", "CRATES_CN_QTY"):
        if c not in crates_p.columns:
            crates_p[c] = 0
        else:
            crates_p[c] = crates_p[c].fillna(0)
    crates_p["CRATES_NET_QTY"] = crates_p["CRATES_INV_QTY"] - crates_p["CRATES_CN_QTY"]
    sheets["crates"] = crates_p[CUST_COLS + ["CRATES_INV_QTY", "CRATES_CN_QTY", "CRATES_NET_QTY"]]

    # ---------- mehul_returns ----------
    meh = load_mehul()
    if meh.empty:
        sheets["mehul_returns"] = pd.DataFrame(columns=CUST_COLS)
    else:
        meh = meh.rename(columns={"CODE": "A/C"})
        meh = attach_customer(meh)
        brand_cols = [c for c in meh.columns if c.endswith("_FRESH") or c.endswith("_EXP")]
        meh_p = meh.groupby(CUST_COLS, as_index=False, dropna=False)[brand_cols].sum()
        sheets["mehul_returns"] = meh_p

    # ---------- hansa_returns ----------
    han = load_hansa()
    han = attach_customer(han)
    product_cols = [c for c in han.columns if norm_brand(c) and c not in ("A/C", "Qty")]
    numeric_products = []
    for c in product_cols:
        han[c] = pd.to_numeric(han[c], errors="coerce").fillna(0)
        numeric_products.append(c)
    han_p = han.groupby(CUST_COLS, as_index=False, dropna=False)[numeric_products + ["Qty"]].sum()
    sheets["hansa_returns"] = han_p

    # ---------- sales_revenue ----------
    s_inv, s_cn = load_sales_revenue()
    s_inv = s_inv.assign(STATUS="Inv")
    s_cn  = s_cn.assign(STATUS="CN")
    sales_all = pd.concat([s_inv, s_cn], ignore_index=True)
    sales_all = sales_all.rename(columns={"a/c_no": "A/C"})
    sales_all = attach_customer(sales_all)
    sales_p = sales_all.groupby(CUST_COLS + ["STATUS"], as_index=False, dropna=False).agg(
        QTY=("qty", "sum"), AMOUNT=("amount", "sum")
    )
    sales_p = sales_p.pivot(
        index=CUST_COLS, columns="STATUS", values=["QTY", "AMOUNT"]
    )
    sales_p.columns = [f"SALES_{stat.upper()}_{m}" for m, stat in sales_p.columns]
    sales_p = sales_p.reset_index()
    for c in ("SALES_INV_QTY", "SALES_CN_QTY", "SALES_INV_AMOUNT", "SALES_CN_AMOUNT"):
        if c not in sales_p.columns:
            sales_p[c] = 0
        else:
            sales_p[c] = sales_p[c].fillna(0)
    sales_p["SALES_PCT_RETURN"] = np.where(
        sales_p["SALES_INV_AMOUNT"] > 0,
        (sales_p["SALES_CN_AMOUNT"] / sales_p["SALES_INV_AMOUNT"]).round(3), 0
    )
    sheets["sales_revenue"] = sales_p[CUST_COLS + [
        "SALES_INV_QTY", "SALES_CN_QTY",
        "SALES_INV_AMOUNT", "SALES_CN_AMOUNT", "SALES_PCT_RETURN"
    ]]

    # ---------- brands_performance ----------
    b_inv_clean, b_cn, _ = load_brands()
    b_inv_clean = b_inv_clean.assign(STATUS="Inv")
    b_cn        = b_cn.assign(STATUS="CN")
    brands_all = pd.concat([b_inv_clean, b_cn], ignore_index=True, sort=False)
    brands_all = attach_customer(brands_all)
    prod_cols = []
    for c in brands_all.columns:
        if c in CUST_COLS + ["Date", "STATUS", "No", "Invoice No.", "A/C"]:
            continue
        if norm_brand(c):
            brands_all[c] = pd.to_numeric(brands_all[c], errors="coerce").fillna(0)
            prod_cols.append(c)
    b_agg = brands_all.groupby(CUST_COLS + ["STATUS"], as_index=False, dropna=False)[prod_cols].sum()
    rename_map = {c: f"BRANDS_{safe_brand_name(norm_brand(c))}" for c in prod_cols}
    b_agg = b_agg.rename(columns=rename_map)
    canon_products = list(rename_map.values())
    b_p = b_agg.pivot(
        index=CUST_COLS, columns="STATUS", values=canon_products
    )
    b_p.columns = [f"{prod}_{stat.upper()}" for prod, stat in b_p.columns]
    b_p = b_p.reset_index()
    inv_cols = [c for c in b_p.columns if c.endswith("_INV")]
    cn_cols  = [c for c in b_p.columns if c.endswith("_CN")]
    b_p[inv_cols + cn_cols] = b_p[inv_cols + cn_cols].fillna(0)
    b_p["BRANDS_INV_QTY"] = b_p[inv_cols].sum(axis=1)
    b_p["BRANDS_CN_QTY"]  = b_p[cn_cols].sum(axis=1)
    b_p["BRANDS_NET_QTY"] = b_p["BRANDS_INV_QTY"] + b_p["BRANDS_CN_QTY"]
    sheets["brands_performance"] = b_p

    with pd.ExcelWriter(out_path, engine="openpyxl") as writer:
        for name, df in sheets.items():
            df.to_excel(writer, sheet_name=name, index=False)
    print(f"[PRECISE] wrote {out_path}")
    for n, df in sheets.items():
        print(f"  {n}: {len(df)} rows x {len(df.columns)} cols")
    return sheets

## Workbook 3 — `OCT-WK3-COMBINED.xlsx`
One sheet. All domains pivoted, credits preserved as their own columns, qty columns renamed per domain to disambiguate.

In [25]:
def build_combined(out_path):
    # Reuse precise builder in memory (no file write)
    import tempfile
    tmp = os.path.join(tempfile.gettempdir(), "_precise_tmp.xlsx")
    precise = build_precise(tmp)
    os.remove(tmp)

    crates = precise["crates"]
    sales  = precise["sales_revenue"]
    brands = precise["brands_performance"]
    hansa  = precise["hansa_returns"]
    mehul  = precise["mehul_returns"]

    base = load_customer_info()
    domain_customers = pd.concat(
        [df[CUST_COLS] for df in (crates, sales, brands, hansa, mehul)],
        ignore_index=True
    ).drop_duplicates()
    unmatched_customers = domain_customers.merge(
        base[CUST_COLS], on=CUST_COLS, how="left", indicator=True
    )
    unmatched_customers = unmatched_customers.loc[
        unmatched_customers["_merge"].eq("left_only"), CUST_COLS
    ]
    base = pd.concat([base, unmatched_customers], ignore_index=True)

    combined = base.merge(crates, on=CUST_COLS, how="left")
    combined = combined.merge(sales,  on=CUST_COLS, how="left")
    combined = combined.merge(brands, on=CUST_COLS, how="left")

    # Hansa: rename Qty -> HANSA_QTY, product cols -> HANSA_<brand>
    hansa = hansa.copy().rename(columns={"Qty": "HANSA_QTY"})
    hansa_ren = {}
    for c in hansa.columns:
        if c in CUST_COLS or c == "HANSA_QTY":
            continue
        key = norm_brand(c)
        if key:
            hansa_ren[c] = f"HANSA_{safe_brand_name(key)}"
    hansa = hansa.rename(columns=hansa_ren)
    combined = combined.merge(hansa, on=CUST_COLS, how="left")

    # Mehul: brand_cols already <BRAND>_FRESH / <BRAND>_EXP -> prefix with MEHUL_
    mehul = mehul.copy()
    mehul_ren = {c: f"MEHUL_{c}" for c in mehul.columns if c not in CUST_COLS}
    mehul = mehul.rename(columns=mehul_ren)
    combined = combined.merge(mehul, on=CUST_COLS, how="left")

    # Fill numeric NaNs
    numeric = combined.select_dtypes(include=[np.number]).columns
    combined[numeric] = combined[numeric].fillna(0)

    # Ratio columns
    combined["BRANDS_PCT_RETURN"] = np.where(
        combined["BRANDS_INV_QTY"] > 0,
        (combined["BRANDS_CN_QTY"].abs() / combined["BRANDS_INV_QTY"]).round(3), 0
    )
    combined["CRATES_PCT_RETURN"] = np.where(
        combined["CRATES_INV_QTY"] > 0,
        (combined["CRATES_CN_QTY"] / combined["CRATES_INV_QTY"]).round(3), 0
    )

    combined = combined.sort_values("A/C").reset_index(drop=True)

    with pd.ExcelWriter(out_path, engine="openpyxl") as writer:
        combined.to_excel(writer, sheet_name="combined", index=False)
    print(f"[COMBINED] wrote {out_path} ({len(combined)} rows x {len(combined.columns)} cols)")
    return combined

## Run everything

In [26]:
datewise_path = os.path.join(OUTPUT_DIR, "OCT-WK3-DATEWISE.xlsx")
precise_path  = os.path.join(OUTPUT_DIR, "OCT-WK3-PRECISE.xlsx")
combined_path = os.path.join(OUTPUT_DIR, "OCT-WK3-COMBINED.xlsx")

datewise_sheets = build_datewise(datewise_path)
precise_sheets  = build_precise(precise_path)
combined_df     = build_combined(combined_path)

print("\nAll three workbooks generated successfully.")

[DATEWISE] wrote report/october\OCT-WK3-DATEWISE.xlsx
  crates: 10076 rows x 10 cols
  mehul_returns: 6812 rows x 40 cols
  hansa_returns: 2890 rows x 28 cols
  sales_revenue: 348136 rows x 18 cols
  brands_performance: 9887 rows x 29 cols
[PRECISE] wrote report/october\OCT-WK3-PRECISE.xlsx
  crates: 889 rows x 9 cols
  mehul_returns: 952 rows x 32 cols
  hansa_returns: 489 rows x 22 cols
  sales_revenue: 990 rows x 11 cols
  brands_performance: 888 rows x 39 cols
[PRECISE] wrote C:\Users\Anthony\AppData\Local\Temp\_precise_tmp.xlsx
  crates: 889 rows x 9 cols
  mehul_returns: 952 rows x 32 cols
  hansa_returns: 489 rows x 22 cols
  sales_revenue: 990 rows x 11 cols
  brands_performance: 888 rows x 39 cols
[COMBINED] wrote report/october\OCT-WK3-COMBINED.xlsx (1214 rows x 91 cols)

All three workbooks generated successfully.


## Sanity check
Quick peek at each output so you can confirm the layout is what you expect.

In [27]:
for name, df in datewise_sheets.items():
    print(f"\n=== DATEWISE / {name} ===")
    print(df.head(3).to_string())


=== DATEWISE / crates ===
        Date STATUS A/C CUSTOMER_NAME CATEGORY     ROUTE_NAME SALES_REP MERCHANDISERS  Invoice No.  QTY
0 2026-10-11    Inv  55     RUIRUNGU      DIST  KAMAKWA ROUTE  Lillian         Unkown     55260394   47
1 2026-10-12    Inv  55     RUIRUNGU      DIST  KAMAKWA ROUTE  Lillian         Unkown     55260395   41
2 2026-10-12    Inv  55     RUIRUNGU      DIST  KAMAKWA ROUTE  Lillian         Unkown     55260396    1

=== DATEWISE / mehul_returns ===
  _source_sheet   A/C              CUSTOMER_NAME CATEGORY   ROUTE_NAME SALES_REP MERCHANDISERS ROUTE_NO  TOTAL_ROUTE ROUTE_NAME_RAW                   NAME_RAW  P_FRESH  P_EXP  4SQ_FRESH  4SQ_EXP  B_FRESH  B_EXP  S_FRESH  S_EXP  XLOONG_FRESH  XLOONG_EXP  200G_FRESH  200G_EXP  O P_FRESH  O P_EXP  800G_FRESH  800G_EXP  600G_FRESH  600G_EXP  MANDO_FRESH  MANDO_EXP  FAMILY_FRESH  FAMILY_EXP  MAICHO_FRESH  MAICHO_EXP  B BITEZ_FRESH  B BITEZ_EXP _IGNORED_27 _IGNORED_28 _IGNORED_29
0          11TH   100                   MAJE

In [28]:
for name, df in precise_sheets.items():
    print(f"\n=== PRECISE / {name} ===")
    print(df.head(3).to_string())


=== PRECISE / crates ===
STATUS   A/C   CUSTOMER_NAME CATEGORY       ROUTE_NAME   SALES_REP MERCHANDISERS  CRATES_INV_QTY  CRATES_CN_QTY  CRATES_NET_QTY
0        100        MAJENGO      DIST      DAKAR SALES    Dorothy         Unkown             405              3             402
1       1000    PIPELINE NRB     DIST  MULOLONGO ROUTE    Dorothy         Unkown             630              1             629
2       1003  NGOINGWA THIKA     DIST         JUJA NEW  Magdaline         Unkown             382              0             382

=== PRECISE / mehul_returns ===
    A/C   CUSTOMER_NAME CATEGORY       ROUTE_NAME   SALES_REP MERCHANDISERS  P_FRESH  P_EXP  4SQ_FRESH  4SQ_EXP  B_FRESH  B_EXP  S_FRESH  S_EXP  XLOONG_FRESH  XLOONG_EXP  200G_FRESH  200G_EXP  O P_FRESH  O P_EXP  800G_FRESH  800G_EXP  600G_FRESH  600G_EXP  MANDO_FRESH  MANDO_EXP  FAMILY_FRESH  FAMILY_EXP  MAICHO_FRESH  MAICHO_EXP  B BITEZ_FRESH  B BITEZ_EXP
0   100        MAJENGO      DIST      DAKAR SALES    Dorothy         

In [29]:
print("\n=== COMBINED ===")
print("Columns:", list(combined_df.columns))
print(combined_df.head(3).to_string())


=== COMBINED ===
Columns: ['A/C', 'CUSTOMER_NAME', 'CATEGORY', 'ROUTE_NAME', 'SALES_REP', 'MERCHANDISERS', 'CRATES_INV_QTY', 'CRATES_CN_QTY', 'CRATES_NET_QTY', 'SALES_INV_QTY', 'SALES_CN_QTY', 'SALES_INV_AMOUNT', 'SALES_CN_AMOUNT', 'SALES_PCT_RETURN', 'BRANDS_200G_CN', 'BRANDS_200G_INV', 'BRANDS_4_SQUARE_CN', 'BRANDS_4_SQUARE_INV', 'BRANDS_50_50_SLICED_CN', 'BRANDS_50_50_SLICED_INV', 'BRANDS_600G_CN', 'BRANDS_600G_INV', 'BRANDS_800G_CN', 'BRANDS_800G_INV', 'BRANDS_B_CN', 'BRANDS_B_INV', 'BRANDS_B_BITEZ_CN', 'BRANDS_B_BITEZ_INV', 'BRANDS_DOUGHNUTS_CN', 'BRANDS_DOUGHNUTS_INV', 'BRANDS_KATIKATI_CN', 'BRANDS_KATIKATI_INV', 'BRANDS_MANDAZI_CN', 'BRANDS_MANDAZI_INV', 'BRANDS_MANDO_FAMILY_PACK_CN', 'BRANDS_MANDO_FAMILY_PACK_INV', 'BRANDS_OP_CN', 'BRANDS_OP_INV', 'BRANDS_P_CN', 'BRANDS_P_INV', 'BRANDS_S_CN', 'BRANDS_S_INV', 'BRANDS_XTRA_LOONG_CN', 'BRANDS_XTRA_LOONG_INV', 'BRANDS_INV_QTY', 'BRANDS_CN_QTY', 'BRANDS_NET_QTY', 'HANSA_P', 'HANSA_S', 'HANSA_B', 'HANSA_200G', 'HANSA_OP', 'HANSA_800